# 🧵 Threads em Python

### Conceitos Teóricos

Antes de codificar, é essencial entender o modelo de concorrência do Python. 

- **Thread**: menor unidade de execução dentro de um processo; compartilha memória com outras threads do mesmo processo 
- **GIL (Global Interpreter Lock)**: mecanismo do Python que permite apenas uma thread executar bytecode Python por vez — threads são úteis para tarefas I/O-bound, mas não ganham paralelismo real em CPU-bound
- **Concorrência ≠ Paralelismo**: threads executam de forma intercalada (concorrente), não simultaneamente em múltiplos núcleos 

```
Tarefa I/O-bound  → threading ✅ (GIL é liberado durante I/O)
Tarefa CPU-bound  → multiprocessing ✅ (processos separados)
```

### Etapa 1 — Criação Básica de Threads

Duas formas de criar threads: via função alvo (`target`) ou subclasse de `Thread`. 

In [ ]:
import threading
import time

# --- Forma 1: via função target ---
def tarefa(nome, duracao):
    print(f"[{nome}] Iniciando...")
    time.sleep(duracao)
    print(f"[{nome}] Concluída após {duracao}s")

t1 = threading.Thread(target=tarefa, args=("T1", 2))
t2 = threading.Thread(target=tarefa, args=("T2", 1), kwargs={})

t1.start()
t2.start()

t1.join()  # bloqueia até T1 terminar
t2.join()  # bloqueia até T2 terminar

print("Programa principal encerrado.")

In [ ]:
# --- Forma 2: subclasse de Thread ---
class MinhaThread(threading.Thread):
    def __init__(self, nome, duracao):
        super().__init__(name=nome)
        self.duracao = duracao
        self.resultado = None

    def run(self):  # sobrescreva run(), nunca start()
        print(f"[{self.name}] Executando...")
        time.sleep(self.duracao)
        self.resultado = f"Resultado de {self.name}"
        print(f"[{self.name}] Finalizada.")

t = MinhaThread("Worker-1", 1.5)
t.start()
t.join()
print(f"Retorno: {t.resultado}")

### Etapa 2 — Ciclo de Vida e Inspeção

Entendendo os estados de uma thread e como inspecioná-los. 

In [ ]:
import threading
import time

def ciclo_longo():
    time.sleep(3)

t = threading.Thread(target=ciclo_longo, name="ThreadCiclo")

print(f"Antes de start()  → is_alive: {t.is_alive()}")  # False
t.start()
print(f"Após start()      → is_alive: {t.is_alive()}")  # True

In [ ]:
# Inspecionando todas as threads ativas
print(f"\nThreads ativas: {threading.active_count()}")
for thread in threading.enumerate():
    print(f"  - {thread.name} | daemon: {thread.daemon}")

In [ ]:
t.join(timeout=5)  # espera no máximo 5 segundos
print(f"\nApós join()       → is_alive: {t.is_alive()}")  # False
print(f"Thread atual: {threading.current_thread().name}")


### Etapa 3 — Threads Daemon

Threads daemon são encerradas automaticamente quando o programa principal termina.


In [ ]:
import threading
import time

def servico_background():
    """Simula um serviço que roda indefinidamente."""
    while True:
        print("[Daemon] Verificando sistema...")
        time.sleep(1)

def tarefa_principal():
    print("[Principal] Processando dados...")
    time.sleep(3)
    print("[Principal] Concluído.")

# Thread daemon: morre com o programa principal
daemon = threading.Thread(target=servico_background, daemon=True, name="Monitor")

# Thread normal: programa aguarda sua conclusão
normal = threading.Thread(target=tarefa_principal, name="Worker")

daemon.start()
normal.start()

normal.join()
# Ao sair daqui, o daemon é encerrado automaticamente
print("Programa encerrando. Daemon será morto.")


> ⚠️ **Cuidado**: nunca use thread daemon para operações que precisem ser finalizadas corretamente (ex: gravar arquivos, fechar conexões).

### Etapa 4 — Race Condition (Problema)

Race condition ocorre quando múltiplas threads acessam e modificam dados compartilhados sem sincronização.

In [ ]:
import threading
import time

contador = 0
N_THREADS = 100
N_ITERACOES = 10000

def incrementar():
    global contador
    for _ in range(N_ITERACOES):
        # PROBLEMA: read-modify-write NÃO é atômico
        valor_local = contador   # lê
        valor_local += 1         # modifica
        time.sleep(0.00001)       # simula atraso, aumentando chance de colisão
        contador = valor_local   # escreve

threads = [threading.Thread(target=incrementar) for _ in range(N_THREADS)]
for t in threads: t.start()
for t in threads: t.join()

esperado = N_THREADS * N_ITERACOES
print(f"Esperado : {esperado}")
print(f"Obtido   : {contador}")
print(f"Perdidos : {esperado - contador}  ← race condition!")

### Etapa 5 — Lock (Mutex)

`Lock` é a solução mais básica para race conditions: garante acesso exclusivo a um recurso.

In [ ]:
import threading
import time

contador = 0
lock = threading.Lock()
N_THREADS = 100
N_ITERACOES = 1000

def incrementar_seguro():
    global contador
    for _ in range(N_ITERACOES):
        with lock:  # acquire() + release() automáticos
            valor_local = contador   # lê
            valor_local += 1         # modifica
            time.sleep(0.00001)       # simula atraso, aumentando chance de colisão
            contador = valor_local   # escreve

threads = [threading.Thread(target=incrementar_seguro) for _ in range(N_THREADS)]
for t in threads: t.start()
for t in threads: t.join()

print(f"Esperado : {N_THREADS * N_ITERACOES}")
print(f"Obtido   : {contador}")
print("✅ Sem race condition!")

# Forma explícita (equivalente ao 'with')
def incrementar_explicito():
    global contador
    lock.acquire()
    try:
        contador += 1
    finally:
        lock.release()  # SEMPRE libere no finally

### Etapa 6 — RLock (Lock Reentrante)

`RLock` permite que a **mesma thread** adquira o lock múltiplas vezes sem se bloquear.

In [ ]:
import threading

rlock = threading.RLock()

def operacao_externa():
    with rlock:
        print(f"[{threading.current_thread().name}] Lock externo adquirido")
        operacao_interna()  # sem RLock, isso causaria deadlock!

def operacao_interna():
    with rlock:  # mesma thread pode readquirir
        print(f"[{threading.current_thread().name}] Lock interno adquirido")

# Demonstração: Lock normal causaria deadlock aqui
lock_normal = threading.Lock()
rlock_demo = threading.RLock()

def demonstra_reentrada():
    with rlock_demo:
        print("Primeira aquisição")
        with rlock_demo:
            print("Segunda aquisição (só funciona com RLock!)")
            operacao_externa()

t = threading.Thread(target=demonstra_reentrada)
t.start()
t.join()


### Etapa 7 — Semaphore (Semáforo)

`Semaphore` limita o número de threads que acessam um recurso simultaneamente.

In [ ]:
import threading
import time
import random

# Simula um pool de 3 conexões de banco de dados
MAX_CONEXOES = 3
semaforo = threading.Semaphore(MAX_CONEXOES)
ativas = 0
lock_print = threading.Lock()

def consultar_banco(thread_id):
    global ativas
    with semaforo:  # bloqueia se já houver 3 threads dentro
        with lock_print:
            ativas += 1
            print(f"Thread-{thread_id:02d} conectou  | Conexões ativas: {ativas}")
        
        time.sleep(random.uniform(0.5, 1.5))  # simula consulta
        
        with lock_print:
            ativas -= 1
            print(f"Thread-{thread_id:02d} desconectou | Conexões ativas: {ativas}")

threads = [threading.Thread(target=consultar_banco, args=(i,)) for i in range(10)]
for t in threads: t.start()
for t in threads: t.join()
print("Todas as consultas concluídas.")


### Etapa 8 — Event (Sinalização entre Threads)

`Event` permite que threads esperem por um sinal antes de continuar.

In [ ]:
import threading
import time

evento_dados_prontos = threading.Event()
evento_processado = threading.Event()

def produtor():
    print("[Produtor] Coletando dados...")
    time.sleep(2)
    print("[Produtor] Dados prontos! Sinalizando consumidor.")
    evento_dados_prontos.set()  # sinaliza o evento
    
    evento_processado.wait()    # aguarda confirmação
    print("[Produtor] Consumidor confirmou processamento.")

def consumidor():
    print("[Consumidor] Aguardando dados...")
    evento_dados_prontos.wait()  # bloqueia até o evento ser sinalizado
    print("[Consumidor] Dados recebidos! Processando...")
    time.sleep(1)
    print("[Consumidor] Processamento concluído.")
    evento_processado.set()

t_prod = threading.Thread(target=produtor, name="Produtor")
t_cons = threading.Thread(target=consumidor, name="Consumidor")

t_cons.start()
t_prod.start()
t_prod.join()
t_cons.join()

# Usando Event como flag de parada
stop_event = threading.Event()

def loop_continuo():
    while not stop_event.is_set():
        print("Executando...")
        time.sleep(0.5)
    print("Loop encerrado por sinal externo.")

t = threading.Thread(target=loop_continuo)
t.start()
time.sleep(2)
stop_event.set()  # sinaliza parada
t.join()

### Etapa 9 — Condition (Produtor-Consumidor)

`Condition` combina Lock com mecanismo de notificação — ideal para o padrão produtor-consumidor.

In [ ]:
import threading
import time
import random
from collections import deque

buffer = deque(maxlen=5)
condition = threading.Condition()

def produtor(n_itens):
    for i in range(n_itens):
        with condition:
            while len(buffer) == buffer.maxlen:
                print("[Produtor] Buffer cheio. Aguardando...")
                condition.wait()  # libera o lock e aguarda
            
            item = f"item-{i}"
            buffer.append(item)
            print(f"[Produtor] Produziu {item} | Buffer: {list(buffer)}")
            condition.notify_all()  # acorda threads esperando
        time.sleep(random.uniform(0.1, 0.5))

def consumidor(nome):
    while True:
        with condition:
            while not buffer:
                condition.wait()
            item = buffer.popleft()
            print(f"[{nome}] Consumiu {item} | Buffer: {list(buffer)}")
            condition.notify_all()
        time.sleep(random.uniform(0.3, 0.8))

# Inicia 1 produtor e 2 consumidores
tp = threading.Thread(target=produtor, args=(10,), daemon=True)
tc1 = threading.Thread(target=consumidor, args=("Consumidor-A",), daemon=True)
tc2 = threading.Thread(target=consumidor, args=("Consumidor-B",), daemon=True)

tc1.start(); tc2.start(); tp.start()
tp.join()
time.sleep(2)
print("Demonstração encerrada.")

### Etapa 10 — Barrier (Ponto de Sincronização)

`Barrier` faz N threads esperarem umas pelas outras antes de prosseguir.

In [ ]:
import threading
import time
import random

N_WORKERS = 4
barreira = threading.Barrier(N_WORKERS)

def trabalhador(id_worker):
    # Fase 1: trabalho independente
    duracao = random.uniform(0.5, 2.0)
    print(f"[W{id_worker}] Fase 1: trabalhando por {duracao:.1f}s")
    time.sleep(duracao)
    
    print(f"[W{id_worker}] Fase 1 concluída. Aguardando na barreira...")
    barreira.wait()  # todas as threads esperam aqui
    
    # Fase 2: só começa após TODOS terminarem a Fase 1
    print(f"[W{id_worker}] Fase 2: iniciando (todas sincronizadas!)")
    time.sleep(0.5)
    print(f"[W{id_worker}] Concluído.")

threads = [threading.Thread(target=trabalhador, args=(i,)) for i in range(N_WORKERS)]
for t in threads: t.start()
for t in threads: t.join()

### Etapa 11 — Queue Thread-Safe

`queue.Queue` é a estrutura de dados ideal para comunicação segura entre threads.

In [ ]:
import threading
import queue
import time
import random

fila = queue.Queue(maxsize=10)
SENTINELA = None  # sinal de fim

def produtor_queue(n_itens):
    for i in range(n_itens):
        item = random.randint(1, 100)
        fila.put(item)  # bloqueia se fila cheia
        print(f"[Produtor] Enfileirou: {item}")
        time.sleep(0.1)
    fila.put(SENTINELA)  # sinaliza fim
    print("[Produtor] Encerrado.")

def consumidor_queue():
    total = 0
    while True:
        item = fila.get()  # bloqueia se fila vazia
        if item is SENTINELA:
            fila.task_done()
            break
        total += item
        print(f"[Consumidor] Processou: {item} | Soma: {total}")
        fila.task_done()  # sinaliza que o item foi processado
    print(f"[Consumidor] Total acumulado: {total}")

tp = threading.Thread(target=produtor_queue, args=(8,))
tc = threading.Thread(target=consumidor_queue)

tp.start(); tc.start()
tp.join(); tc.join()

fila.join()  # aguarda todos os task_done()
print("Fila completamente processada.")

### Etapa 12 — ThreadPoolExecutor

`concurrent.futures.ThreadPoolExecutor` simplifica o gerenciamento de múltiplas threads.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import random

def baixar_arquivo(url):
    """Simula download de arquivo (I/O-bound)."""
    duracao = random.uniform(0.5, 2.0)
    time.sleep(duracao)
    tamanho = random.randint(100, 5000)
    return {"url": url, "tamanho_kb": tamanho, "tempo_s": round(duracao, 2)}

urls = [f"https://servidor.com/arquivo_{i}.csv" for i in range(8)]

print("=== Com ThreadPoolExecutor ===")
inicio = time.time()

# Forma 1: map() — mantém ordem dos resultados
with ThreadPoolExecutor(max_workers=4) as executor:
    resultados = list(executor.map(baixar_arquivo, urls))

for r in resultados:
    print(f"  {r['url']}: {r['tamanho_kb']} KB em {r['tempo_s']}s")

print(f"Tempo total: {time.time() - inicio:.2f}s")

# Forma 2: submit() + as_completed() — processa conforme terminam
print("\n=== Com as_completed() ===")
inicio = time.time()

with ThreadPoolExecutor(max_workers=4) as executor:
    futuros = {executor.submit(baixar_arquivo, url): url for url in urls}
    
    for futuro in as_completed(futuros):
        url = futuros[futuro]
        try:
            resultado = futuro.result()
            print(f"  ✅ {url}: {resultado['tamanho_kb']} KB")
        except Exception as e:
            print(f"  ❌ {url}: erro — {e}")

print(f"Tempo total: {time.time() - inicio:.2f}s")

### Etapa 13 — Thread-Local Data

`threading.local()` cria dados que são **exclusivos de cada thread**, evitando compartilhamento acidental.

In [ ]:
import threading
import time

# Dados locais por thread (cada thread tem sua própria cópia)
dados_locais = threading.local()

def inicializar_conexao(nome_thread):
    """Simula uma conexão de BD exclusiva por thread."""
    dados_locais.conexao = f"Conexão-{nome_thread}"
    dados_locais.usuario = nome_thread
    dados_locais.contador = 0

def processar(nome_thread):
    inicializar_conexao(nome_thread)
    
    for i in range(3):
        dados_locais.contador += 1
        time.sleep(0.1)
    
    print(f"[{nome_thread}] Usando {dados_locais.conexao} | "
          f"Usuário: {dados_locais.usuario} | "
          f"Ops: {dados_locais.contador}")

threads = [
    threading.Thread(target=processar, args=(f"Thread-{i}",))
    for i in range(4)
]
for t in threads: t.start()
for t in threads: t.join()
# Cada thread tem seu próprio contador — sem interferência!

### Etapa 14 — Deadlock (Demonstração e Prevenção)

Deadlock ocorre quando duas threads esperam uma pela outra, causando bloqueio eterno.

In [ ]:
import threading
import time

lock_a = threading.Lock()
lock_b = threading.Lock()

# ❌ DEADLOCK — NÃO EXECUTE por muito tempo
def thread_1_deadlock():
    with lock_a:
        print("[T1] Adquiriu lock_a, aguardando lock_b...")
        time.sleep(0.1)
        with lock_b:  # T2 já tem lock_b → DEADLOCK
            print("[T1] Adquiriu lock_b")

def thread_2_deadlock():
    with lock_b:
        print("[T2] Adquiriu lock_b, aguardando lock_a...")
        time.sleep(0.1)
        with lock_a:  # T1 já tem lock_a → DEADLOCK
            print("[T2] Adquiriu lock_a")

# ✅ SOLUÇÃO 1: Ordenação consistente de locks
def thread_1_segura():
    with lock_a:  # sempre adquire A antes de B
        with lock_b:
            print("[T1 Segura] Adquiriu A → B")

def thread_2_segura():
    with lock_a:  # mesma ordem: A antes de B
        with lock_b:
            print("[T2 Segura] Adquiriu A → B")

# ✅ SOLUÇÃO 2: timeout com acquire()
def thread_com_timeout(nome):
    adquiriu_a = lock_a.acquire(timeout=1.0)
    if not adquiriu_a:
        print(f"[{nome}] Timeout em lock_a. Abortando.")
        return
    try:
        adquiriu_b = lock_b.acquire(timeout=1.0)
        if not adquiriu_b:
            print(f"[{nome}] Timeout em lock_b. Liberando lock_a.")
            return
        try:
            print(f"[{nome}] Ambos os locks adquiridos com sucesso!")
        finally:
            lock_b.release()
    finally:
        lock_a.release()

print("=== Solução com ordenação ===")
t1 = threading.Thread(target=thread_1_segura)
t2 = threading.Thread(target=thread_2_segura)
t1.start(); t2.start()
t1.join(); t2.join()

### Etapa 15 — GIL: CPU-bound vs I/O-bound

Demonstração prática do impacto do GIL em tarefas CPU-bound vs I/O-bound. 

In [ ]:
import threading
import time
import math

# --- CPU-bound: threads NÃO ajudam (GIL limita) ---
def tarefa_cpu(n):
    """Calcula raiz quadrada de N números (CPU puro)."""
    return sum(math.sqrt(i) for i in range(n))

N = 5_000_000

# Sequencial
inicio = time.perf_counter()
tarefa_cpu(N)
tarefa_cpu(N)
print(f"CPU-bound Sequencial: {time.perf_counter() - inicio:.2f}s")

# Com threads (não melhora em CPU-bound)
inicio = time.perf_counter()
t1 = threading.Thread(target=tarefa_cpu, args=(N,))
t2 = threading.Thread(target=tarefa_cpu, args=(N,))
t1.start(); t2.start()
t1.join(); t2.join()
print(f"CPU-bound 2 Threads : {time.perf_counter() - inicio:.2f}s ← sem ganho!")

# --- I/O-bound: threads AJUDAM (GIL liberado durante I/O) ---
def tarefa_io(duracao):
    """Simula operação de I/O."""
    time.sleep(duracao)  # GIL é liberado durante sleep

DURACAO = 1.0

# Sequencial
inicio = time.perf_counter()
tarefa_io(DURACAO); tarefa_io(DURACAO)
print(f"\nI/O-bound Sequencial: {time.perf_counter() - inicio:.2f}s")

# Com threads (melhora em I/O-bound!)
inicio = time.perf_counter()
t1 = threading.Thread(target=tarefa_io, args=(DURACAO,))
t2 = threading.Thread(target=tarefa_io, args=(DURACAO,))
t1.start(); t2.start()
t1.join(); t2.join()
print(f"I/O-bound 2 Threads : {time.perf_counter() - inicio:.2f}s ← ~2x mais rápido!")

## Resumo dos Primitivos

| Primitivo | Uso principal | Quando usar |
|---|---|---|
| `Lock` | Exclusão mútua | Proteger recursos compartilhados simples |
| `RLock` | Lock reentrante | Quando a mesma thread precisa reentrar |
| `Semaphore` | Limite de acesso | Pool de conexões, recursos limitados |
| `Event` | Sinalização | Notificar início/fim de evento |
| `Condition` | Produtor-consumidor | Coordenação com condições variáveis |
| `Barrier` | Sincronização em fases | Aguardar todos chegarem a um ponto |
| `Queue` | Comunicação segura | Passar dados entre threads  [docs.python](https://docs.python.org/3/library/threading.html) |
| `ThreadPoolExecutor` | Gerenciar pool | Simplificar criação de múltiplas threads  [realpython](https://realpython.com/intro-to-python-threading/) |
| `local()` | Dados por thread | Evitar compartilhamento acidental |